# Evaluate ranges of output dataset
Diagnostics comparing our downscaled data against the ranges from ERA5. Rather than hard-and-fast checks for aphysical values (e.g. negative precip), this checks for squishier situations like is there a flood in the Sahara. Goal is to surface problems. 

In [1]:
import math

import boto3
import icechunk
import matplotlib.pyplot as plt
import xarray as xr
from srm.utils import resolve_s3_glob, open_icechunk
from srm import catalog

In [12]:
def get_fname(var, scenario, version="test015_benchmark", ens="003"):
    fname = (
        "s3://carbonplan-srm/output/qa/"
        + version
        + "/"
        + scenario
        + "/CESM2-WACCM/"
        + var
        + "/"
        + ens
        + "/lat-35.0to-22.0_lon16.0to33.0/gdex-gmf-icechunk/*/"
        + scenario
        + ".icechunk/"
    )

    return fname

In [3]:
era5 = catalog.get("ERA5").to_xarray()

In [4]:
_MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def calc_era5_threshold(era5, downscaled_da, var, direction, period="annual"):
    """Per-pixel ERA5 threshold: 'gt' → max, 'lt' → min; 'monthly' groups by calendar month."""
    era5_da = era5[var].sel(lat=downscaled_da.lat, lon=downscaled_da.lon, method="nearest")
    if period == "annual":
        if direction == "gt":
            return era5_da.max(dim="time").load()
        return era5_da.min(dim="time").load()
    if direction == "gt":
        return era5_da.groupby("time.month").max(dim="time").load()
    return era5_da.groupby("time.month").min(dim="time").load()


def count_exceedances(downscaled_da, threshold, direction, period="annual"):
    """Annual: (lat, lon) count over all time. Monthly: (month, lat, lon) count per calendar month."""
    if period == "annual":
        if direction == "gt":
            return (downscaled_da > threshold).sum(dim="time").load()
        return (downscaled_da < threshold).sum(dim="time").load()
    counts = []
    for m, group in downscaled_da.groupby("time.month"):
        thresh = threshold.sel(month=m)
        exceeded = group > thresh if direction == "gt" else group < thresh
        counts.append(exceeded.sum("time").expand_dims(month=[m]))
    return xr.concat(counts, dim="month").load()


def plot_exceedance_map(count_da, title, period="annual"):
    """Single panel for annual; 3×4 panel for monthly (one per calendar month)."""
    if period == "annual":
        fig, ax = plt.subplots(figsize=(8, 6))
        count_da.plot(ax=ax, cmap="Reds")
        ax.set_title(title)
        plt.tight_layout()
        return fig
    fig, axs = plt.subplots(nrows=3, ncols=4, figsize=(16, 12))
    fig.suptitle(title, fontsize=12)
    for i, ax in enumerate(axs.flatten()):
        m = i + 1
        count_da.sel(month=m).plot(ax=ax, cmap="Reds")
        ax.set_title(_MONTH_NAMES[i])
    plt.tight_layout()
    return fig

In [16]:
era5.lat

<xarray.DataArray 'lat' (lat: 721)> Size: 3kB
array([-90.  , -89.75, -89.5 , ...,  89.5 ,  89.75,  90.  ],
      shape=(721,), dtype=float32)
Coordinates:
  * lat      (lat) float32 3kB -90.0 -89.75 -89.5 -89.25 ... 89.5 89.75 90.0
Attributes:
    long_name:  latitude
    units:      degrees_north
    bounds:     lat_bounds

In [17]:
downscaled_da.lat

<xarray.DataArray 'lat' (lat: 52)> Size: 416B
array([-34.875, -34.625, -34.375, -34.125, -33.875, -33.625, -33.375, -33.125,
       -32.875, -32.625, -32.375, -32.125, -31.875, -31.625, -31.375, -31.125,
       -30.875, -30.625, -30.375, -30.125, -29.875, -29.625, -29.375, -29.125,
       -28.875, -28.625, -28.375, -28.125, -27.875, -27.625, -27.375, -27.125,
       -26.875, -26.625, -26.375, -26.125, -25.875, -25.625, -25.375, -25.125,
       -24.875, -24.625, -24.375, -24.125, -23.875, -23.625, -23.375, -23.125,
       -22.875, -22.625, -22.375, -22.125])
Coordinates:
  * lat      (lat) float64 416B -34.88 -34.62 -34.38 ... -22.62 -22.38 -22.12
Attributes:
    units:      degrees_north
    long_name:  Latitude
    bounds:     lat_bounds

In [13]:
version = "obs_comparison"
scenario = "ssp245"
ens = "003"

checks = [
    # (var, direction, period, title)
    # ("tasmax", "gt", "annual",  "Greater than ERA5 annual max tasmax"),
    # ("tasmax", "gt", "monthly", "Greater than ERA5 monthly max tasmax"),
    # ("tasmin", "lt", "annual",  "Less than ERA5 annual min tasmin"),
    # ("tasmin", "lt", "monthly", "Less than ERA5 monthly min tasmin"),
    ("tas",    "gt", "annual",  "Greater than ERA5 annual max tasmean"),
    ("tas",    "lt", "annual",  "Less than ERA5 annual min tasmean"),
    # ("pr",     "gt", "monthly", "Greater than ERA5 monthly max precipitation"),
    # ("pr",     "gt", "annual",  "Greater than ERA5 annual max precipitation"),
    # ("rsds",   "gt", "annual",  "Greater than ERA5 annual max rsds"),
    # ("hurs",   "lt", "annual",  "Less than ERA5 annual min relative humidity"),
]

for var, direction, period, title in checks:
    fname = get_fname(var=var, scenario=scenario, version=version, ens=ens)
    print(fname)
    fname = resolve_s3_glob(fname)
    downscaled_da = open_icechunk(path=fname)[var]

    threshold = calc_era5_threshold(era5, downscaled_da, var, direction, period)
    # count = count_exceedances(downscaled_da, threshold, direction, period)
    # plot_exceedance_map(count, title, period=period)

s3://carbonplan-srm/output/qa/obs_comparison/ssp245/CESM2-WACCM/tas/003/lat-35.0to-22.0_lon16.0to33.0/gdex-gmf-icechunk/*/ssp245.icechunk/


In [9]:
# from code

# from raphael
s3://carbonplan-srm/output/qa/obs_comparison/ssp245/CESM2-WACCM/tas/003/lat-35.0to-22.0_lon16.0to33.0/gdex-gmf-icechunk/5e6b6f6c/ssp245.icechunk

SyntaxError: invalid decimal literal (3923415030.py, line 1)